In [1]:

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from scipy.integrate import solve_ivp
from tqdm import tqdm
import os

# ─────────────────────────────────────────────────────────────────────────────
# GLOBAL CONSTANTS  (edit these to scale up / down)
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
rng  = np.random.default_rng(seed=SEED)

# Strain library size.  The paper uses 100 strains (88 in the earliest phase).
N_STRAINS = 100

# Pathogen is always index 0 in the internal species array.
PATHOGEN_IDX = 0
N_COMMUNITY_STRAINS = N_STRAINS  # non-pathogen library members

# How many communities to generate.  Paper has ~1,500.
N_COMMUNITIES = 1500

# Community size distribution: uniform between MIN and MAX.
# Paper: sizes 2 to 48 (we use 50 as the ceiling).
COMMUNITY_SIZE_MIN = 2
COMMUNITY_SIZE_MAX = 50

# MiCRM resource model
N_RESOURCES  = 20   # nutrient dimensions in the ODE
N_GENES_REAL = 8    # functional gene features (uptake-linked)

# Gene-orthologue feature matrix dimensions
# Paper starts with ~58,764 and reduces to ~572.
# We simulate 1,000 GO features; ~5–15 % non-zero per strain.
N_GO_FEATURES = 1000

# Output directory — change to your Downloads folder
OUTPUT_DIR = r"C:\Users\Pratham Shah\Downloads\Document dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Strain type splits (library is all non-pathogen)
FRAC_SPECIALIST = 0.40   # 40 % specialists, 60 % generalists
N_SPECIALIST = int(N_STRAINS * FRAC_SPECIALIST)
N_GENERALIST = N_STRAINS - N_SPECIALIST

# Uptake sparsity: what fraction of nutrients each type can consume
UPTAKE_SPARSITY = {
    "pathogen"  : (0.30, 0.55),
    "specialist": (0.10, 0.25),   # specialists focus on 1–5 nutrients
    "generalist": (0.40, 0.65),   # generalists consume broadly
}

CROSSFEED_OUT_MIN = 2
CROSSFEED_OUT_MAX = 6

RESOURCE_NAMES = [f"c{k+1}" for k in range(N_RESOURCES)]

# Luminescence: we treat it as log(pathogen_abundance / reference).
# Suppression score = clip(1 - N_path_comm / N_path_solo, 0, 1)

TYPE_COLORS = {
    "pathogen"  : "#E53935",
    "specialist": "#1E88E5",
    "generalist": "#43A047",
}


# =============================================================================
# SECTION 1 — BUILD STRAIN LIBRARY TRAITS
# =============================================================================

def build_strain_library():
    """
    Creates the fixed 100-strain library.

    Returns
    -------
    genes       : ndarray [N_STRAINS, N_GENES_REAL]   binary genome
    c_matrix    : ndarray [N_STRAINS+1, N_RESOURCES]  uptake rates (row 0 = pathogen)
    uptake_mask : ndarray [N_STRAINS+1, N_RESOURCES]  boolean mask
    m           : ndarray [N_STRAINS+1]               maintenance costs
    g           : ndarray [N_STRAINS+1]               max growth rates
    D_list      : list[ndarray]  per-species cross-feeding matrices (N_STRAINS+1)
    strain_type : list[str]      'pathogen' | 'specialist' | 'generalist'
    preferred_nutrient : list[int]  preferred nutrient index (specialists only)
    go_matrix   : ndarray [N_STRAINS, N_GO_FEATURES]  gene-orthologue feature matrix

    NOTE: The pathogen is row 0 in c_matrix / D_list etc. but is NOT in
          go_matrix (library strains only, indices 1..N_STRAINS internally).
    """
    # Total "species" in the ODE world = pathogen (1) + library (N_STRAINS)
    N_TOTAL = N_STRAINS + 1   # internal index 0 = pathogen

    # ── Functional genes (binary) ────────────────────────────────────────────
    # genes shape [N_TOTAL, N_GENES_REAL]. Row 0 = pathogen (also has genes).
    genes = rng.integers(0, 2, size=(N_TOTAL, N_GENES_REAL))

    # Weight matrix: maps gene presence to nutrient uptake affinity.
    # Shape [N_GENES_REAL, N_RESOURCES].
    W = rng.normal(0.0, 0.5, size=(N_GENES_REAL, N_RESOURCES))

    c_matrix    = np.zeros((N_TOTAL, N_RESOURCES))
    uptake_mask = np.zeros((N_TOTAL, N_RESOURCES), dtype=bool)
    strain_type = []
    preferred_nutrient = []

    for i in range(N_TOTAL):
        # Assign type
        if i == PATHOGEN_IDX:
            stype = "pathogen"
            pref  = -1
        elif 1 <= i <= N_SPECIALIST:
            stype = "specialist"
            # Each specialist favours one nutrient, cycling through resources
            # so different specialists prefer different nutrients.
            pref = (i - 1) % N_RESOURCES
        else:
            stype = "generalist"
            pref  = -1

        strain_type.append(stype)
        preferred_nutrient.append(pref)

        # ── Sparsity mask ────────────────────────────────────────────────────
        # Draw how many nutrients this strain can consume.
        lo, hi  = UPTAKE_SPARSITY[stype]
        density = rng.uniform(lo, hi)
        # max(1, ...) ensures at least one active nutrient.
        n_active = max(1, int(round(density * N_RESOURCES)))

        # Choose which nutrients are active (without replacement).
        active_indices = rng.choice(N_RESOURCES, size=n_active, replace=False).tolist()

        # Specialists must have their preferred nutrient in the active set.
        if pref >= 0 and pref not in active_indices:
            active_indices[0] = pref   # overwrite the first slot

        uptake_mask[i, active_indices] = True

        # ── Compute sigmoid uptake rates ─────────────────────────────────────
        # Each strain's uptake rate for nutrient k comes from its gene profile
        # passed through a linear-then-sigmoid transform.
        # Specialists get an extra bias on their preferred nutrient so they are
        # measurably better at consuming it.
        if stype == "specialist":
            bias = rng.uniform(-2.0, -1.0, size=N_RESOURCES)
            # Add a large positive bias on the preferred nutrient.
            bias[pref] += rng.uniform(2.5, 4.0)
        else:
            bias = rng.uniform(-0.8, -0.3, size=N_RESOURCES)

        # Linear combination of gene presence + bias → sigmoid → uptake rate.
        # sigmoid(x) = 1 / (1 + exp(-x)) squashes values into (0, 1).
        raw    = genes[i].astype(float) @ W + bias
        c_full = 1.0 / (1.0 + np.exp(-raw))
        # Zero out nutrients outside the sparsity mask.
        c_matrix[i] = c_full * uptake_mask[i]

    # ── Maintenance costs and growth rates ───────────────────────────────────
    # m[i] = metabolic cost a strain must cover before it can grow.
    # g[i] = conversion efficiency (max growth rate scalar).
    m = rng.uniform(0.2, 0.6, size=N_TOTAL)
    g = rng.uniform(0.8, 1.2, size=N_TOTAL)

    # ── Cross-feeding matrices ────────────────────────────────────────────────
    # D_list[i] is an [N_RESOURCES, N_RESOURCES] matrix.
    # D_i[alpha, beta] = fraction of nutrient beta leakage from species i
    #                    that becomes nutrient alpha (cross-feeding product).
    # Each column is sparse: only CROSSFEED_OUT_MIN to CROSSFEED_OUT_MAX
    # output slots are non-zero, and they sum to 1 (mass conservation).
    D_list = _build_crossfeed_matrices(N_TOTAL, strain_type)

    # ── Gene-orthologue feature matrix ───────────────────────────────────────
    # This is the "genomic" feature used in GAT-GO.
    # We simulate N_GO_FEATURES orthologue copy-number columns.
    # Each strain has ~5–15 % non-zero entries (sparse genome coverage).
    # Copy numbers are small integers (0, 1, 2, occasionally 3).
    # The GO matrix is only for the library strains (not the pathogen).
    go_matrix = _build_go_matrix(N_STRAINS, genes[1:])   # exclude pathogen row

    print(f"  Strain library built:")
    print(f"    Pathogen + library total (ODE): {N_TOTAL}")
    print(f"    Library (non-pathogen): {N_STRAINS}  "
          f"({N_SPECIALIST} specialists, {N_GENERALIST} generalists)")
    active_counts = uptake_mask[1:].sum(axis=1)   # library only
    print(f"    Mean active nutrients: {active_counts.mean():.1f}  "
          f"(min={active_counts.min()}, max={active_counts.max()})")
    print(f"    GO feature matrix shape: {go_matrix.shape}  "
          f"(sparsity: {(go_matrix == 0).mean():.1%} zeros)")

    return genes, c_matrix, uptake_mask, m, g, D_list, strain_type, preferred_nutrient, go_matrix


def _build_crossfeed_matrices(N_TOTAL, strain_type):
    """
    One [N_RESOURCES, N_RESOURCES] cross-feeding matrix per species.
    Column beta gives the output distribution when species i consumes
    nutrient beta — sparse Dirichlet, sums to 1 per column.
    """
    D_list = []
    for i in range(N_TOTAL):
        D_i = np.zeros((N_RESOURCES, N_RESOURCES))
        for beta in range(N_RESOURCES):
            n_out = int(rng.integers(CROSSFEED_OUT_MIN, CROSSFEED_OUT_MAX + 1))
            active_alpha = rng.choice(N_RESOURCES, size=n_out, replace=False)

            # Dirichlet concentration parameters determine output evenness.
            # Specialists channel leakage preferentially to their preferred nutrient.
            if strain_type[i] == "specialist":
                pref = (i - 1) % N_RESOURCES
                alpha_vec = np.ones(n_out)
                for j, a in enumerate(active_alpha):
                    if a == pref and beta != pref:
                        alpha_vec[j] = rng.uniform(2.0, 5.0)
                    else:
                        alpha_vec[j] = rng.uniform(0.5, 2.0)
            elif strain_type[i] == "pathogen":
                alpha_vec = rng.uniform(0.5, 3.0, size=n_out)
            else:
                alpha_vec = rng.uniform(0.8, 1.5, size=n_out)

            # rng.dirichlet draws a probability vector that sums to 1.
            probs = rng.dirichlet(alpha_vec)
            D_i[active_alpha, beta] = probs
        D_list.append(D_i)
    return D_list


def _build_go_matrix(n_strains, functional_genes):
    """
    Simulate a gene-orthologue copy-number matrix.

    Real GO matrices are very high-dimensional (tens of thousands of columns)
    and very sparse. We create N_GO_FEATURES columns where:
      - Each column has ~5–15 % non-zero entries across strains
        (mimicking ~5–15 strains out of 100 carrying a given orthologue)
      - Non-zero values are copy numbers: mostly 1, occasionally 2 or 3
      - A fraction of GO features are correlated with functional genes
        (so the model can learn signal from genomes)

    Parameters
    ----------
    n_strains       : int, number of library strains (100)
    functional_genes: ndarray [n_strains, N_GENES_REAL] — used to inject signal

    Returns
    -------
    go_matrix : ndarray [n_strains, N_GO_FEATURES]  — integer copy numbers
    """
    go_matrix = np.zeros((n_strains, N_GO_FEATURES), dtype=np.int8)

    for f in range(N_GO_FEATURES):
        # What fraction of strains carry this orthologue?
        # Draw from a beta distribution biased toward low frequency.
        # beta(0.5, 4) gives values mostly in [0.02, 0.30].
        frac = rng.beta(0.5, 4.0)
        n_carriers = max(1, int(round(frac * n_strains)))

        carriers = rng.choice(n_strains, size=n_carriers, replace=False)

        # Copy number: 1 most of the time, 2 or 3 occasionally.
        # rng.choice([1,1,1,2,3], size=...) gives copy-number distribution.
        copy_nums = rng.choice([1, 1, 1, 2, 3], size=n_carriers)
        go_matrix[carriers, f] = copy_nums

        # Inject signal: for ~20 % of GO features, correlate with a functional gene.
        # This ensures that gene-orthologue-based models can learn something real.
        if rng.random() < 0.20:
            linked_gene = int(rng.integers(0, N_GENES_REAL))
            # Strains with the linked functional gene are twice as likely to carry
            # this GO feature.
            has_gene = np.where(functional_genes[:, linked_gene] == 1)[0]
            extra_carriers = rng.choice(
                has_gene,
                size=min(len(has_gene) // 4, max(1, n_carriers // 2)),
                replace=False
            )
            go_matrix[extra_carriers, f] = np.maximum(
                go_matrix[extra_carriers, f], 1
            )

    return go_matrix


# =============================================================================
# SECTION 2 — NUTRIENT MEDIA (replaces "broths" — now a single shared glucose
#             medium with variations, matching the paper's minimal media setup)
# =============================================================================

def build_growth_medium():
    """
    Build one shared nutrient environment.

    The paper uses minimal media + NM + glucose (one carbon source).
    We model this as a single R0 vector (initial resource concentrations)
    with one dominant nutrient (glucose analogue) and background nutrients
    available at lower concentrations.

    Returns
    -------
    R0    : ndarray [N_RESOURCES]  initial resource concentrations
    w     : ndarray [N_RESOURCES]  energy yield per unit resource consumed
    l     : ndarray [N_RESOURCES]  leakage fraction (for cross-feeding)
    delta : float                  dilution rate (chemostat)
    """
    # One dominant resource (glucose) at high concentration.
    dominant = int(rng.integers(0, N_RESOURCES))

    R0 = np.zeros(N_RESOURCES)
    for k in range(N_RESOURCES):
        if k == dominant:
            # Dominant nutrient: high supply rate (glucose-like)
            R0[k] = rng.uniform(0.5, 2.0) * rng.uniform(3.0, 6.0)
        else:
            # Background nutrients: low supply (cross-feeding substrates)
            if rng.random() < 0.15:   # 15 % chance of being absent
                R0[k] = 0.0
            else:
                R0[k] = rng.uniform(0.1, 0.8)

    w     = rng.uniform(0.6, 1.2, size=N_RESOURCES)   # energy yield
    l     = rng.uniform(0.1, 0.3, size=N_RESOURCES)   # leakage fraction
    delta = float(rng.uniform(0.05, 0.15))             # dilution / wash-out rate

    return R0, w, l, delta, dominant


# =============================================================================
# SECTION 3 — ODE (MiCRM)
# =============================================================================

def micrm_ode(t, y, c, w, l, m, g, delta, R0, D_list_sub):
    """
    MacArthur Consumer-Resource Model ODE (right-hand side).

    State vector y = [N_1, ..., N_n, R_1, ..., R_K]
      N_i: abundance of species i
      R_k: concentration of resource k

    Equations:
      dN_i/dt = N_i * [g_i * (sum_k c_ik * w_k * (1-l_k) * R_k - m_i)] - delta * N_i
      dR_k/dt = delta*(R0_k - R_k) - (sum_i c_ik * N_i) * R_k
                + sum_i sum_beta D_i[k,beta] * (w_beta/w_k) * l_beta * c_i_beta * N_i * R_beta

    Parameters (passed as args via solve_ivp):
      c          : [n_sp, n_res]  uptake rate matrix for this subset of species
      w          : [n_res]        energy yield per resource unit
      l          : [n_res]        leakage fraction (cross-feeding source)
      m          : [n_sp]         maintenance cost per species
      g          : [n_sp]         growth efficiency per species
      delta      : float          dilution rate
      R0         : [n_res]        resource supply concentrations
      D_list_sub : list of [n_res, n_res]  cross-feeding matrices for subset
    """
    n_sp  = c.shape[0]
    n_res = c.shape[1]
    # np.maximum clips negative abundances/concentrations to 0 (physical constraint)
    N = np.maximum(y[:n_sp], 0.0)
    R = np.maximum(y[n_sp:], 0.0)

    # Energy intake per species: how much each species gains from available resources
    energy_intake = c @ (w * (1.0 - l) * R)   # shape [n_sp]
    # Net growth: growth from energy intake minus maintenance cost
    net_growth    = g * (energy_intake - m)     # shape [n_sp]
    # Population dynamics: growth - dilution (chemostat washout)
    dN = N * net_growth - delta * N

    # Resource dynamics: supply (from medium) - uptake (consumed by species)
    dilution_net = delta * (R0 - R)             # resource input from medium
    uptake       = (c.T @ N) * R                # total uptake from all species

    # Cross-feeding: leaked nutrients from consumption become new resources
    eps = 1e-12
    crossfeed = np.zeros(n_res)
    for i in range(n_sp):
        if N[i] < 1e-14:
            continue    # skip extinct species (saves computation)
        for beta in range(n_res):
            if c[i, beta] < 1e-14:
                continue
            # Flux of nutrient beta leaking from species i
            flux_i_beta = l[beta] * c[i, beta] * N[i] * R[beta]
            if flux_i_beta < 1e-16:
                continue
            # Distribute this flux to output nutrients alpha via D_i
            col = D_list_sub[i][:, beta]
            nz  = np.nonzero(col)[0]   # only iterate non-zero outputs (sparse)
            for alpha in nz:
                crossfeed[alpha] += (col[alpha]
                                     * w[beta] / (w[alpha] + eps)
                                     * flux_i_beta)

    dR = dilution_net - uptake + crossfeed
    return np.concatenate([dN, dR])


# =============================================================================
# SECTION 4 — ODE RUNNER
# =============================================================================

def run_micrm(c_matrix, w, l, m, g, delta, R0, D_list,
              species_subset, t_end=300.0):
    """
    Integrate the MiCRM ODE for a specific subset of species.

    Parameters
    ----------
    c_matrix      : [N_TOTAL, N_RESOURCES]  full uptake matrix (all species)
    species_subset: list[int]               global indices of species in this run
                                            (index 0 = pathogen)
    t_end         : float                   integration time horizon

    Returns
    -------
    N_out   : ndarray [N_TOTAL]  steady-state abundances (0 for absent species)
    R_final : ndarray [N_RESOURCES]  final resource concentrations
    """
    # Slice the full matrices to the subset of species in this run.
    # This is important — running all 101 species at once is much slower.
    c_sub      = c_matrix[species_subset, :]
    m_sub      = m[species_subset]
    g_sub      = g[species_subset]
    D_list_sub = [D_list[i] for i in species_subset]
    n_sub      = len(species_subset)

    # Initial conditions: all species start at low abundance (0.01),
    # resources start at their supply concentrations.
    y0 = np.concatenate([np.full(n_sub, 0.01), R0.copy()])

    # solve_ivp: scipy's ODE integrator.
    # method="RK45" = Runge-Kutta 4th/5th order (adaptive step size)
    # rtol, atol = relative and absolute error tolerances
    sol = solve_ivp(
        fun    = micrm_ode,
        t_span = (0.0, t_end),
        y0     = y0,
        args   = (c_sub, w, l, m_sub, g_sub, delta, R0, D_list_sub),
        method = "RK45",
        t_eval = np.linspace(0, t_end, 200),   # 200 time points for diagnostics
        rtol   = 1e-5,
        atol   = 1e-7,
    )

    y_final = sol.y[:, -1]                          # take final time point
    N_ss    = np.maximum(y_final[:n_sub], 0.0)      # steady-state abundances
    R_final = np.maximum(y_final[n_sub:], 0.0)

    # Map local indices back to global species indices.
    # N_out is full-size but zeros for species not in this run.
    N_total = c_matrix.shape[0]
    N_out   = np.zeros(N_total)
    for local_idx, global_idx in enumerate(species_subset):
        N_out[global_idx] = N_ss[local_idx]

    return N_out, R_final


# =============================================================================
# SECTION 5 — LUMINESCENCE AND SUPPRESSION METRICS
# =============================================================================

def abundance_to_luminescence(N_pathogen, reference=1.0):
    """
    Convert pathogen abundance to log-luminescence.

    In the paper, luminescence is measured as log(signal / reference).
    Values are ≤ 0 (lower = more suppression).
    We use log10 for interpretability.

    Parameters
    ----------
    N_pathogen : float  pathogen steady-state abundance from ODE
    reference  : float  reference abundance (pathogen solo in same medium)

    Returns
    -------
    lum : float  log10(N_pathogen / reference), capped at 0
    """
    eps = 1e-10
    # log10(ratio): 0 means no suppression; very negative = strong suppression
    lum = np.log10((N_pathogen + eps) / (reference + eps))
    return float(lum)


def calc_suppression_score(N_pathogen_comm, N_pathogen_solo):
    """
    Normalised suppression score in [0, 1].

    suppression = 1 - (N_path_comm / N_path_solo)
    Clipped to [0, 1]: species/communities that facilitate the pathogen score 0.
    """
    eps = 1e-10
    if N_pathogen_solo < eps:
        return 0.0
    raw = 1.0 - (N_pathogen_comm / (N_pathogen_solo + eps))
    return float(np.clip(raw, 0.0, 1.0))


# =============================================================================
# SECTION 6 — MAIN DATASET GENERATOR
# =============================================================================

def generate_dataset():
    """
    Full pipeline:
      1. Build strain library (traits + GO features)
      2. Build growth medium
      3. Run monoculture ODE for each library strain (single-strain baseline)
      4. Run monoculture ODE for pathogen alone (reference luminescence)
      5. Sample 1,500 communities of varying sizes
      6. Run community ODE for each
      7. Assemble output DataFrames

    Returns
    -------
    df_community  : pd.DataFrame  member-level rows (for ML)
    df_mono       : pd.DataFrame  single-strain data
    df_comm_index : pd.DataFrame  community-level labels
    """

    # ── Step 1: Strain library ────────────────────────────────────────────────
    print("\n[1/6] Building strain library …")
    (genes, c_matrix, uptake_mask, m, g,
     D_list, strain_type, preferred_nutrient, go_matrix) = build_strain_library()

    # Internal indices: 0 = pathogen, 1..N_STRAINS = library strains.
    # Library strain s (0-indexed) = internal index s+1.
    library_indices = list(range(1, N_STRAINS + 1))   # internal ODE indices
    strain_names    = [f"Strain_{s+1:03d}" for s in range(N_STRAINS)]

    # ── Step 2: Growth medium ─────────────────────────────────────────────────
    print("[2/6] Building growth medium …")
    R0, w, l, delta, dominant_nutrient_idx = build_growth_medium()
    print(f"    Dominant nutrient: {RESOURCE_NAMES[dominant_nutrient_idx]}  "
          f"  delta={delta:.3f}")

    # ── Step 3: Pathogen solo (reference) ─────────────────────────────────────
    print("[3/6] Running pathogen solo ODE (reference luminescence) …")
    N_path_solo, _ = run_micrm(
        c_matrix, w, l, m, g, delta, R0, D_list,
        species_subset=[PATHOGEN_IDX], t_end=300.0
    )
    pathogen_solo_abundance = float(N_path_solo[PATHOGEN_IDX])
    pathogen_solo_lum       = 0.0   # log(solo/solo) = log(1) = 0 by definition
    print(f"    Pathogen solo abundance: {pathogen_solo_abundance:.4f}  "
          f"lum (reference) = {pathogen_solo_lum:.3f}")

    # ── Step 4: Monoculture ODE for each library strain ───────────────────────
    print("[4/6] Running monoculture ODE for each library strain …")
    mono_rows = []

    for s_idx, (internal_idx, sname) in enumerate(
            zip(library_indices, strain_names)):
        # Run pathogen + this one strain together (co-culture, no community).
        # This gives the single-strain suppression ability = SS lum in paper.
        N_pair, _ = run_micrm(
            c_matrix, w, l, m, g, delta, R0, D_list,
            species_subset=[internal_idx, PATHOGEN_IDX], t_end=300.0
        )
        N_path_pair = float(N_pair[PATHOGEN_IDX])
        N_strain    = float(N_pair[internal_idx])

        ss_lum       = abundance_to_luminescence(N_path_pair, pathogen_solo_abundance)
        ss_supp      = calc_suppression_score(N_path_pair, pathogen_solo_abundance)

        row = {
            "strain_id"        : sname,
            "strain_type"      : strain_type[internal_idx],
            "preferred_nutrient": (RESOURCE_NAMES[preferred_nutrient[internal_idx]]
                                   if preferred_nutrient[internal_idx] >= 0 else "none"),
            "maintenance_cost" : round(float(m[internal_idx]), 4),
            "growth_rate"      : round(float(g[internal_idx]), 4),
            "delta"            : round(delta, 4),
            "dominant_nutrient": RESOURCE_NAMES[dominant_nutrient_idx],
        }
        # Functional genes (binary)
        for gi in range(N_GENES_REAL):
            row[f"gene_{gi+1}"] = int(genes[internal_idx, gi])
        # Uptake rates per resource
        for k in range(N_RESOURCES):
            row[f"uptake_{RESOURCE_NAMES[k]}"] = round(float(c_matrix[internal_idx, k]), 4)
        # Resource supply concentrations (same for all strains — environment)
        for k in range(N_RESOURCES):
            row[f"R0_{RESOURCE_NAMES[k]}"] = round(float(R0[k]), 4)
        # Single-strain phenotype (the key feature for GAT-SS)
        row["ss_abundance"]         = round(N_strain, 4)
        row["ss_luminescence"]      = round(ss_lum, 4)       # ≤ 0
        row["ss_suppression_score"] = round(ss_supp, 4)      # [0, 1]
        # GO features (gene orthologues)
        for f in range(N_GO_FEATURES):
            row[f"GO_{f+1:05d}"] = int(go_matrix[s_idx, f])

        mono_rows.append(row)

    df_mono = pd.DataFrame(mono_rows)
    print(f"    Monoculture dataset: {df_mono.shape}")
    print(f"    SS lum range: [{df_mono['ss_luminescence'].min():.3f}, "
          f"{df_mono['ss_luminescence'].max():.3f}]")
    print(f"    Fraction suppressing (score > 0): "
          f"{(df_mono['ss_suppression_score'] > 0).mean():.1%}")

    # ── Step 5 & 6: Sample communities and run ODE ────────────────────────────
    print(f"\n[5&6/6] Generating {N_COMMUNITIES} communities (sizes {COMMUNITY_SIZE_MIN}–{COMMUNITY_SIZE_MAX}) …")

    # We want approximately equal numbers of communities at each size class,
    # matching the paper's design ("approximately equal numbers of communities
    # of each strength").
    # np.linspace then round gives roughly equal counts across sizes.
    community_sizes = np.round(
        np.random.default_rng(SEED).integers(
            COMMUNITY_SIZE_MIN, COMMUNITY_SIZE_MAX + 1, size=N_COMMUNITIES
        )
    ).astype(int)

    community_rows  = []   # one row per (community, member strain)
    comm_index_rows = []   # one row per community

    for comm_idx in tqdm(range(N_COMMUNITIES), desc="  Communities", unit="comm"):
        comm_size    = int(community_sizes[comm_idx])
        comm_name    = f"Comm_{comm_idx+1:05d}"

        # Sample which library strains are in this community (no replacement).
        # rng.choice is WITHOUT replacement by default here: each strain appears
        # at most once per community, just like real experimental assembly.
        member_s_indices = rng.choice(N_STRAINS, size=comm_size, replace=False)
        # Convert to internal ODE indices (library strain s → index s+1)
        member_internal  = [int(idx) + 1 for idx in member_s_indices]

        # Always include the pathogen in the ODE run.
        species_subset = [PATHOGEN_IDX] + member_internal

        # Run community ODE.
        N_comm, _ = run_micrm(
            c_matrix, w, l, m, g, delta, R0, D_list,
            species_subset=species_subset, t_end=300.0
        )

        N_path_comm  = float(N_comm[PATHOGEN_IDX])
        comm_lum     = abundance_to_luminescence(N_path_comm, pathogen_solo_abundance)
        comm_supp    = calc_suppression_score(N_path_comm, pathogen_solo_abundance)

        # Community-level index row
        comm_index_rows.append({
            "community_id"             : comm_name,
            "community_size"           : comm_size,
            "community_luminescence"   : round(comm_lum,  4),    # ≤ 0
            "community_suppression"    : round(comm_supp, 4),    # [0, 1]
            "pathogen_abundance_comm"  : round(N_path_comm, 6),
            "pathogen_abundance_solo"  : round(pathogen_solo_abundance, 6),
        })

        # Per-member rows (one per strain in community)
        for s_local_idx, (internal_idx, s_idx) in enumerate(
                zip(member_internal, member_s_indices)):
            sname   = strain_names[s_idx]
            N_strain = float(N_comm[internal_idx])

            row = {
                "community_id"             : comm_name,
                "community_size"           : comm_size,
                "strain_id"                : sname,
                "strain_type"              : strain_type[internal_idx],
                "dominant_nutrient"        : RESOURCE_NAMES[dominant_nutrient_idx],
                "preferred_nutrient"       : (RESOURCE_NAMES[preferred_nutrient[internal_idx]]
                                              if preferred_nutrient[internal_idx] >= 0
                                              else "none"),
                "maintenance_cost"         : round(float(m[internal_idx]), 4),
                "growth_rate"              : round(float(g[internal_idx]), 4),
                "delta"                    : round(delta, 4),
            }
            # Functional genes
            for gi in range(N_GENES_REAL):
                row[f"gene_{gi+1}"] = int(genes[internal_idx, gi])
            # Uptake rates
            for k in range(N_RESOURCES):
                row[f"uptake_{RESOURCE_NAMES[k]}"] = round(float(c_matrix[internal_idx, k]), 4)
            # Resource supply
            for k in range(N_RESOURCES):
                row[f"R0_{RESOURCE_NAMES[k]}"] = round(float(R0[k]), 4)
            # Single-strain phenotype (from monoculture data — this is the NODE FEATURE
            # in GAT-SS, and does not require knowing community composition)
            mono_row = df_mono.iloc[s_idx]
            row["ss_luminescence"]      = mono_row["ss_luminescence"]
            row["ss_suppression_score"] = mono_row["ss_suppression_score"]
            row["strain_abundance_in_community"] = round(N_strain, 6)
            # Community-level label (same for every member of this community)
            row["community_luminescence"] = round(comm_lum,  4)
            row["community_suppression"]  = round(comm_supp, 4)
            # GO features
            for f in range(N_GO_FEATURES):
                row[f"GO_{f+1:05d}"] = int(go_matrix[s_idx, f])

            community_rows.append(row)

    df_community  = pd.DataFrame(community_rows)
    df_comm_index = pd.DataFrame(comm_index_rows)

    # ── Summary stats ─────────────────────────────────────────────────────────
    print(f"\n  ┌─ Dataset Summary ─────────────────────────────────────┐")
    print(f"  │ Community member rows  : {len(df_community):>8,}                     │")
    print(f"  │ Communities            : {N_COMMUNITIES:>8,}                     │")
    print(f"  │ Unique strains         : {df_community['strain_id'].nunique():>8,}                     │")
    print(f"  │ Community size range   : {df_community['community_size'].min():>3} – {df_community['community_size'].max():<3}                    │")
    print(f"  │ Community lum range    : [{df_community['community_luminescence'].min():.3f}, {df_community['community_luminescence'].max():.3f}]           │")
    print(f"  │ Fraction comm_supp > 0 : {(df_community['community_suppression'] > 0).mean():.1%}                      │")
    print(f"  │ Monoculture rows       : {len(df_mono):>8,}                     │")
    print(f"  │ GO feature columns     : {N_GO_FEATURES:>8,}                     │")
    print(f"  └───────────────────────────────────────────────────────┘")

    return df_community, df_mono, df_comm_index


# =============================================================================
# SECTION 7 — SAVE
# =============================================================================

def save_datasets(df_community, df_mono, df_comm_index):
    """
    Save the three output files.

    Files
    -----
    community_dataset.csv   — main ML table (one row per strain per community)
                              Columns include both GAT-SS features (ss_luminescence)
                              and GAT-GO features (GO_00001 … GO_01000)
    monoculture_dataset.csv — single-strain data (matched baseline)
    community_index.csv     — community-level labels (suppression, lum, size)
    """
    # Split GO columns out of community_dataset to keep file manageable.
    # The community_dataset_noGO.csv is what most ML pipelines will load first.
    go_cols     = [c for c in df_community.columns if c.startswith("GO_")]
    non_go_cols = [c for c in df_community.columns if not c.startswith("GO_")]

    path_comm        = os.path.join(OUTPUT_DIR, "community_dataset.csv")
    path_comm_nogo   = os.path.join(OUTPUT_DIR, "community_dataset_noGO.csv")
    path_mono        = os.path.join(OUTPUT_DIR, "monoculture_dataset.csv")
    path_comm_idx    = os.path.join(OUTPUT_DIR, "community_index.csv")

    df_community.to_csv(path_comm,     index=False)
    df_community[non_go_cols].to_csv(path_comm_nogo, index=False)
    df_mono.to_csv(path_mono,          index=False)
    df_comm_index.to_csv(path_comm_idx, index=False)

    print(f"\n  Saved files to: {OUTPUT_DIR}")
    print(f"    community_dataset.csv         {df_community.shape}  (full with GO)")
    print(f"    community_dataset_noGO.csv    {df_community[non_go_cols].shape}  (no GO cols)")
    print(f"    monoculture_dataset.csv       {df_mono.shape}")
    print(f"    community_index.csv           {df_comm_index.shape}")


# =============================================================================
# SECTION 8 — RUN
# =============================================================================

if __name__ == "__main__":
    print("=" * 65)
    print("  SYNTHETIC COMMUNITY DATASET GENERATOR")
    print("  (mirrors experimental design of draft_gnns_for_communities)")
    print("=" * 65)
    print(f"  Strain library   : {N_STRAINS} strains")
    print(f"  Communities      : {N_COMMUNITIES} (sizes {COMMUNITY_SIZE_MIN}–{COMMUNITY_SIZE_MAX})")
    print(f"  GO features      : {N_GO_FEATURES}")
    print(f"  ODE resources    : {N_RESOURCES}")
    print(f"  Output directory : {OUTPUT_DIR}")
    print("=" * 65)

    df_community, df_mono, df_comm_index = generate_dataset()
    save_datasets(df_community, df_mono, df_comm_index)

    print("\n  Sample community rows:")
    preview_cols = [
        "community_id", "community_size", "strain_id", "strain_type",
        "ss_luminescence", "ss_suppression_score",
        "community_luminescence", "community_suppression",
    ]
    print(df_community[preview_cols].head(12).to_string(index=False))

    print("\n  Community index (first 10):")
    print(df_comm_index.head(10).to_string(index=False))

    print("\n  Community luminescence distribution by size bin:")
    df_comm_index["size_bin"] = pd.cut(
        df_comm_index["community_size"],
        bins=[1, 4, 8, 15, 24, 50],
        labels=["2-4", "5-8", "9-15", "16-24", "25-50"]
    )
    print(df_comm_index.groupby("size_bin", observed=True)["community_luminescence"]
          .agg(["mean", "std", "count"])
          .round(3).to_string())
    print("\n[Done]")

  SYNTHETIC COMMUNITY DATASET GENERATOR
  (mirrors experimental design of draft_gnns_for_communities)
  Strain library   : 100 strains
  Communities      : 1500 (sizes 2–50)
  GO features      : 1000
  ODE resources    : 20
  Output directory : C:\Users\Pratham Shah\Downloads\Document dataset

[1/6] Building strain library …
  Strain library built:
    Pathogen + library total (ODE): 101
    Library (non-pathogen): 100  (40 specialists, 60 generalists)
    Mean active nutrients: 7.6  (min=2, max=13)
    GO feature matrix shape: (100, 1000)  (sparsity: 88.2% zeros)
[2/6] Building growth medium …
    Dominant nutrient: c17    delta=0.136
[3/6] Running pathogen solo ODE (reference luminescence) …
    Pathogen solo abundance: 0.4834  lum (reference) = 0.000
[4/6] Running monoculture ODE for each library strain …
    Monoculture dataset: (100, 1058)
    SS lum range: [-9.684, 0.016]
    Fraction suppressing (score > 0): 59.0%

[5&6/6] Generating 1500 communities (sizes 2–50) …


  Communities: 100%|███████████████████████████████████████████████████████████| 1500/1500 [1:01:27<00:00,  2.46s/comm]



  ┌─ Dataset Summary ─────────────────────────────────────┐
  │ Community member rows  :   39,362                     │
  │ Communities            :    1,500                     │
  │ Unique strains         :      100                     │
  │ Community size range   :   2 – 50                     │
  │ Community lum range    : [-9.684, 0.002]           │
  │ Fraction comm_supp > 0 : 99.9%                      │
  │ Monoculture rows       :      100                     │
  │ GO feature columns     :    1,000                     │
  └───────────────────────────────────────────────────────┘

  Saved files to: C:\Users\Pratham Shah\Downloads\Document dataset
    community_dataset.csv         (39362, 1062)  (full with GO)
    community_dataset_noGO.csv    (39362, 62)  (no GO cols)
    monoculture_dataset.csv       (100, 1058)
    community_index.csv           (1500, 6)

  Sample community rows:
community_id  community_size  strain_id strain_type  ss_luminescence  ss_suppression_score  comm

In [8]:
"""
community_figures.py
=====================
Add this to your community_dataset_generator.py
Call make_figures() at the end of your __main__ block, after save_datasets().

Generates 7 figures that let you visually inspect what was generated:

  Fig 1 — Strain Library Overview        (who are the strains?)
  Fig 2 — Single-Strain Suppression      (how good is each strain alone?)
  Fig 3 — Community Suppression vs Size  (the key biological result)
  Fig 4 — Outcome Distribution by Size   (emergent simplicity / variance compression)
  Fig 5 — Uptake Rate Heatmap            (which strains eat which nutrients?)
  Fig 6 — GO Feature Sparsity            (what does the genome data look like?)
  Fig 7 — Community Member Breakdown     (who ends up in which communities?)

Usage:
------
  from community_figures import make_figures
  make_figures(df_community, df_mono, df_comm_index, go_matrix,
               c_matrix, strain_type, OUTPUT_DIR)
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — saves to file, no popup window
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
import os

# ─── Colour palette (consistent across all figures) ──────────────────────────
# We define named colours so every figure uses the same scheme.
# If you want to change the look, just change these.
PAL = {
    "specialist" : "#1E88E5",   # blue
    "generalist" : "#43A047",   # green
    "pathogen"   : "#E53935",   # red
    "highlight"  : "#FF8F00",   # amber — used for annotations / markers
    "neutral"    : "#78909C",   # grey-blue — used for baselines / reference lines
    "bg"         : "#FAFAFA",   # near-white background
    "text"       : "#212121",   # near-black text
}

# Global matplotlib style — applied once, affects all figures
plt.rcParams.update({
    "figure.facecolor"  : PAL["bg"],
    "axes.facecolor"    : "white",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.labelcolor"   : PAL["text"],
    "xtick.color"       : PAL["text"],
    "ytick.color"       : PAL["text"],
    "font.family"       : "sans-serif",
    "font.size"         : 11,
    "axes.titlesize"    : 12,
    "axes.titleweight"  : "bold",
})


# =============================================================================
# FIGURE 1 — Strain Library Overview
# =============================================================================

def fig1_strain_library(df_mono, OUTPUT_DIR):
    """
    Three-panel figure showing what the 100-strain library looks like:
      Panel A: How many specialists vs generalists
      Panel B: Maintenance cost distribution by type
      Panel C: Growth rate vs maintenance cost scatter (coloured by type)

    Why does this matter?
    You want to sanity-check that the library was built correctly —
    the right proportion of types, and that maintenance costs and growth
    rates look reasonable (not all identical, not wildly extreme).
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Figure 1 — Strain Library Overview (100 strains)",
                 fontsize=14, fontweight="bold", color=PAL["text"])

    # ── Panel A: Strain type count (bar chart) ────────────────────────────────
    # value_counts() counts how many rows have each unique value in strain_type.
    # sort_index() sorts alphabetically so the order is consistent.
    type_counts = df_mono["strain_type"].value_counts().sort_index()

    bar_colors = [PAL.get(t, PAL["neutral"]) for t in type_counts.index]
    bars = axes[0].bar(type_counts.index, type_counts.values,
                       color=bar_colors, edgecolor="white", linewidth=1.5,
                       width=0.5)

    # Add the count number on top of each bar for easy reading.
    # bar.get_height() returns the bar's y value (the count).
    # axes.text() places text at coordinates (x, y) on the plot.
    for bar in bars:
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.5,
                     str(int(bar.get_height())),
                     ha="center", va="bottom",
                     fontweight="bold", color=PAL["text"])

    axes[0].set_title("A) Strain Type Composition")
    axes[0].set_xlabel("Strain Type")
    axes[0].set_ylabel("Number of Strains")
    axes[0].set_ylim(0, max(type_counts.values) * 1.15)
    axes[0].grid(True, axis="y", alpha=0.3)

    # ── Panel B: Maintenance cost distribution (overlapping histograms) ────────
    # We plot one histogram per strain type, overlapping, with alpha=0.6
    # (60% transparent) so both are visible where they overlap.
    for stype in ["specialist", "generalist"]:
        subset = df_mono[df_mono["strain_type"] == stype]["maintenance_cost"]
        # bins=15 means divide the data range into 15 equal-width buckets.
        axes[1].hist(subset, bins=15, color=PAL[stype], alpha=0.6,
                     label=f"{stype.capitalize()} (n={len(subset)})",
                     edgecolor="none")

    axes[1].set_title("B) Maintenance Cost Distribution")
    axes[1].set_xlabel("Maintenance cost m_i\n(energy needed just to survive)")
    axes[1].set_ylabel("Number of strains")
    axes[1].legend(fontsize=9)
    axes[1].grid(True, axis="y", alpha=0.3)

    # ── Panel C: Growth rate vs maintenance cost scatter ──────────────────────
    # A scatter plot with one dot per strain, coloured by type.
    # This reveals whether growth rate and maintenance cost are correlated
    # (they shouldn't be, since they were drawn independently — good sanity check).
    for stype in ["specialist", "generalist"]:
        subset = df_mono[df_mono["strain_type"] == stype]
        # axes.scatter() draws individual points.
        # s=50 is the marker size, alpha=0.7 makes them slightly transparent.
        # edgecolors="white" adds a white border to each dot for separation.
        axes[2].scatter(subset["maintenance_cost"], subset["growth_rate"],
                        c=PAL[stype], s=55, alpha=0.75,
                        edgecolors="white", linewidths=0.5,
                        label=stype.capitalize())

    axes[2].set_title("C) Growth Rate vs Maintenance Cost")
    axes[2].set_xlabel("Maintenance cost m_i")
    axes[2].set_ylabel("Growth rate g_i")
    axes[2].legend(fontsize=9)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "fig1_strain_library.png")
    # dpi=150 gives a reasonably high-resolution image without being huge.
    # bbox_inches="tight" trims whitespace around the figure.
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)   # close to free memory — important in a script generating many figures
    print(f"  → fig1_strain_library.png")


# =============================================================================
# FIGURE 2 — Single-Strain Suppression (the monoculture result)
# =============================================================================

def fig2_single_strain_suppression(df_mono, OUTPUT_DIR):
    """
    Four-panel figure showing how well each strain suppresses the pathogen alone:
      Panel A: Distribution of ss_luminescence (the raw readout)
      Panel B: Ranked bar chart — best to worst suppressors
      Panel C: ss_luminescence by strain type (box plot)
      Panel D: ss_suppression_score vs maintenance cost

    Why does this matter?
    This is the key input feature for GAT-SS — the single-strain luminescence
    is what each GNN node carries. You want to see that there is real variation
    (some strains much better than others), not all the same.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Figure 2 — Single-Strain Suppression Ability (monoculture data)",
                 fontsize=14, fontweight="bold", color=PAL["text"])

    # ── Panel A: Distribution of ss_luminescence ──────────────────────────────
    # ss_luminescence is log10(N_pathogen_with_strain / N_pathogen_alone).
    # Values cluster at 0 (no suppression) or go negative (suppression).
    for stype in ["specialist", "generalist"]:
        subset = df_mono[df_mono["strain_type"] == stype]["ss_luminescence"]
        axes[0, 0].hist(subset, bins=20, color=PAL[stype], alpha=0.6,
                        label=stype.capitalize(), edgecolor="none")

    # axvline draws a vertical line at x=0 (the reference: no suppression).
    axes[0, 0].axvline(0, color=PAL["neutral"], lw=1.5, ls="--",
                       label="No suppression (reference)")
    axes[0, 0].set_title("A) Distribution of Single-Strain Luminescence")
    axes[0, 0].set_xlabel("ss_luminescence  [log₁₀(ratio)]\n"
                          "← more negative = stronger suppressor")
    axes[0, 0].set_ylabel("Number of strains")
    axes[0, 0].legend(fontsize=9)
    axes[0, 0].grid(True, axis="y", alpha=0.3)

    # ── Panel B: Ranked bar chart of all 100 strains ──────────────────────────
    # sort_values() sorts the DataFrame by ss_luminescence from lowest to highest.
    # reset_index(drop=True) gives new sequential row numbers after sorting.
    ranked = df_mono.sort_values("ss_luminescence").reset_index(drop=True)
    bar_colors_ranked = [PAL[t] for t in ranked["strain_type"]]

    axes[0, 1].bar(range(len(ranked)), ranked["ss_luminescence"],
                   color=bar_colors_ranked, edgecolor="none", alpha=0.85)
    axes[0, 1].axhline(0, color=PAL["neutral"], lw=1.2, ls="--")

    # Annotate the top 5 suppressors (most negative = best at suppression).
    # We label the first 5 bars (lowest ss_luminescence values).
    for rank_pos in range(min(5, len(ranked))):
        row = ranked.iloc[rank_pos]
        axes[0, 1].annotate(
            row["strain_id"].replace("Strain_", "S"),
            xy=(rank_pos, row["ss_luminescence"]),
            xytext=(rank_pos + 3, row["ss_luminescence"] - 0.1),
            fontsize=7, color=PAL["highlight"],
            arrowprops=dict(arrowstyle="->", color=PAL["highlight"], lw=0.8)
        )

    axes[0, 1].set_title("B) All Strains Ranked by Suppression Ability")
    axes[0, 1].set_xlabel("Strain rank (0 = best suppressor)")
    axes[0, 1].set_ylabel("ss_luminescence")

    # Custom legend for strain types
    legend_handles = [
        mpatches.Patch(color=PAL["specialist"], label="Specialist"),
        mpatches.Patch(color=PAL["generalist"], label="Generalist"),
    ]
    axes[0, 1].legend(handles=legend_handles, fontsize=9)
    axes[0, 1].grid(True, axis="y", alpha=0.3)

    # ── Panel C: Box plot — ss_luminescence by type ───────────────────────────
    # A box plot shows the median, quartiles, and outliers at once.
    # It's better than a histogram when comparing groups because both groups
    # fit on the same axis with the same scale.
    # We use seaborn's boxplot for a cleaner look.
    sns.boxplot(
        data=df_mono, x="strain_type", y="ss_luminescence",
        palette={"specialist": PAL["specialist"], "generalist": PAL["generalist"]},
        width=0.5, linewidth=1.5,
        flierprops=dict(marker="o", markersize=4, alpha=0.5),
        ax=axes[1, 0]
    )
    # Overlay individual data points as a strip plot for transparency.
    # A strip plot shows where each actual data point is, which the box hides.
    sns.stripplot(
        data=df_mono, x="strain_type", y="ss_luminescence",
        palette={"specialist": PAL["specialist"], "generalist": PAL["generalist"]},
        size=4, alpha=0.4, jitter=True, ax=axes[1, 0]
    )
    axes[1, 0].axhline(0, color=PAL["neutral"], lw=1.2, ls="--",
                       label="No suppression")
    axes[1, 0].set_title("C) Suppression by Strain Type")
    axes[1, 0].set_xlabel("Strain Type")
    axes[1, 0].set_ylabel("ss_luminescence")
    axes[1, 0].legend(fontsize=9)
    axes[1, 0].grid(True, axis="y", alpha=0.3)

    # ── Panel D: Suppression score vs maintenance cost ────────────────────────
    # Does having a high maintenance cost hurt suppression ability?
    # This is a biological sanity check — high maintenance = more resources
    # needed just to survive = less competitive against the pathogen.
    for stype in ["specialist", "generalist"]:
        subset = df_mono[df_mono["strain_type"] == stype]
        axes[1, 1].scatter(subset["maintenance_cost"], subset["ss_suppression_score"],
                           c=PAL[stype], s=55, alpha=0.75,
                           edgecolors="white", linewidths=0.5,
                           label=stype.capitalize())

    axes[1, 1].set_title("D) Suppression Score vs Maintenance Cost")
    axes[1, 1].set_xlabel("Maintenance cost m_i")
    axes[1, 1].set_ylabel("ss_suppression_score\n(0 = no suppression, 1 = full suppression)")
    axes[1, 1].legend(fontsize=9)
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "fig2_single_strain_suppression.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  → fig2_single_strain_suppression.png")


# =============================================================================
# FIGURE 3 — Community Suppression vs Community Size (THE key result)
# =============================================================================

def fig3_community_vs_size(df_comm_index, OUTPUT_DIR):
    """
    This is Figure 1B from the paper — the single most important plot.
    It shows that larger communities suppress the pathogen more reliably.

    Panel A: Scatter plot — every community as one dot, size on x, lum on y
    Panel B: Binned summary — mean ± std per size bin (cleaner version of A)
    Panel C: Suppression score version (0–1 scale, more intuitive)

    Why does this matter?
    This is the biological claim: "larger communities are more suppressive."
    If this plot shows the trend, your simulation is working correctly.
    If it's flat or goes the wrong way, something is wrong with the ODE.
    """
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Figure 3 — Community Suppression vs Community Size\n"
                 "(reproduces paper Figure 1B)",
                 fontsize=14, fontweight="bold", color=PAL["text"])

    # ── Panel A: Raw scatter ──────────────────────────────────────────────────
    # Every dot is one community. The cloud should slope downward left-to-right
    # (more negative luminescence = more suppression as size increases).
    # c=df_comm_index["community_size"] colours each dot by its size.
    # cmap="viridis_r" is a colour map (purple=small, yellow=large).
    sc = axes[0].scatter(
        df_comm_index["community_size"],
        df_comm_index["community_luminescence"],
        c=df_comm_index["community_size"],
        cmap="viridis_r", s=15, alpha=0.4, edgecolors="none"
    )
    plt.colorbar(sc, ax=axes[0], label="Community size (N)")
    axes[0].axhline(0, color=PAL["neutral"], lw=1.2, ls="--",
                    label="Pathogen solo (reference)")
    axes[0].set_title("A) All Communities (raw scatter)")
    axes[0].set_xlabel("Community size (number of strains)")
    axes[0].set_ylabel("Community luminescence\n[log₁₀(ratio) — lower = more suppression]")
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.25)

    # ── Panel B: Binned mean ± std ────────────────────────────────────────────
    # pd.cut() divides the continuous community_size into discrete bins.
    # This is called "binning" — instead of showing 1,500 individual points,
    # we summarise each size range as mean ± standard deviation.
    # This is much easier to read and mirrors the paper's presentation.
    bins   = [1, 4, 8, 15, 24, 35, 50]
    labels = ["2-4", "5-8", "9-15", "16-24", "25-35", "36-50"]

    df_comm_index["size_bin"] = pd.cut(
        df_comm_index["community_size"],
        bins=bins, labels=labels
    )

    # groupby().agg() groups rows by size_bin and computes mean, std, count.
    # observed=True suppresses a pandas warning about unused categories.
    binned = (df_comm_index
              .groupby("size_bin", observed=True)["community_luminescence"]
              .agg(["mean", "std", "count"])
              .reset_index())

    # x positions: use the numeric index of each bin (0, 1, 2, ...) for spacing.
    x_pos = np.arange(len(binned))

    # axes.bar() draws the mean as a bar.
    axes[1].bar(x_pos, binned["mean"], color=PAL["specialist"],
                alpha=0.75, edgecolor="white", linewidth=1.2, width=0.6)

    # axes.errorbar() overlays ± std as error bars on each bar.
    # fmt="none" means draw only the error bars, no line connecting them.
    # capsize=5 adds a small horizontal cap at the end of each error bar.
    axes[1].errorbar(x_pos, binned["mean"], yerr=binned["std"],
                     fmt="none", color=PAL["text"], lw=1.5, capsize=5)

    # Annotate each bar with sample count (n=XX).
    for i, row in binned.iterrows():
        axes[1].text(x_pos[i], row["mean"] - row["std"] - 0.05,
                     f"n={int(row['count'])}", ha="center", fontsize=8,
                     color=PAL["neutral"])

    axes[1].axhline(0, color=PAL["neutral"], lw=1.2, ls="--")
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(labels)
    axes[1].set_title("B) Binned Mean ± Std Dev by Size Class")
    axes[1].set_xlabel("Community size bin")
    axes[1].set_ylabel("Mean community luminescence")
    axes[1].grid(True, axis="y", alpha=0.3)

    # ── Panel C: Same thing but using suppression_score (0–1 scale) ───────────
    # Some people find "suppression_score" (0=no suppression, 1=full suppression)
    # more intuitive than log-luminescence.
    binned_supp = (df_comm_index
                   .groupby("size_bin", observed=True)["community_suppression"]
                   .agg(["mean", "std"])
                   .reset_index())

    axes[2].plot(x_pos, binned_supp["mean"],
                 color=PAL["specialist"], lw=2.5, marker="o",
                 markersize=8, markerfacecolor="white",
                 markeredgecolor=PAL["specialist"], markeredgewidth=2,
                 label="Mean suppression")

    # fill_between shades the area between mean-std and mean+std.
    # This gives a visual sense of spread without cluttering the plot.
    axes[2].fill_between(x_pos,
                         binned_supp["mean"] - binned_supp["std"],
                         binned_supp["mean"] + binned_supp["std"],
                         alpha=0.15, color=PAL["specialist"])

    axes[2].set_xticks(x_pos)
    axes[2].set_xticklabels(labels)
    axes[2].set_ylim(-0.05, 1.05)
    axes[2].set_title("C) Suppression Score by Size Class\n(more intuitive 0–1 scale)")
    axes[2].set_xlabel("Community size bin")
    axes[2].set_ylabel("Community suppression score\n(0 = none, 1 = full)")
    axes[2].legend(fontsize=9)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "fig3_community_vs_size.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  → fig3_community_vs_size.png")


# =============================================================================
# FIGURE 4 — Outcome Distribution by Size (Variance Compression)
# =============================================================================

def fig4_variance_compression(df_comm_index, OUTPUT_DIR):
    """
    The paper's key statistical observation: as community size increases,
    the VARIANCE in suppression compresses (outcomes become more similar).

    Panel A: Violin plots — luminescence distribution per size bin
    Panel B: Standard deviation vs community size (should decrease)
    Panel C: 10th, 50th, 90th percentile trends

    Why does this matter?
    This is "emergent predictability" — large communities all converge toward
    similar suppression regardless of which strains are in them. This is why
    the GNN becomes easier to train on larger communities.
    """
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Figure 4 — Outcome Distribution Compression With Diversity\n"
                 "(emergent predictability: larger communities become more similar)",
                 fontsize=14, fontweight="bold", color=PAL["text"])

    bins   = [1, 4, 8, 15, 24, 35, 50]
    labels = ["2-4", "5-8", "9-15", "16-24", "25-35", "36-50"]
    df_comm_index["size_bin"] = pd.cut(
        df_comm_index["community_size"], bins=bins, labels=labels
    )

    # ── Panel A: Violin plots ─────────────────────────────────────────────────
    # A violin plot is like a box plot but also shows the shape of the
    # distribution — it's mirrored on both sides, wider where more data points
    # are concentrated. You should see the violins getting narrower as size
    # increases (less spread in outcomes).
    sns.violinplot(
        data=df_comm_index, x="size_bin", y="community_luminescence",
        color=PAL["specialist"], alpha=0.7, linewidth=1.2,
        inner="quartile",   # draw quartile lines inside the violin
        ax=axes[0]
    )
    axes[0].axhline(0, color=PAL["neutral"], lw=1.2, ls="--")
    axes[0].set_title("A) Outcome Distribution per Size Bin\n"
                      "(violins should get narrower = more predictable)")
    axes[0].set_xlabel("Community size bin")
    axes[0].set_ylabel("Community luminescence")
    axes[0].grid(True, axis="y", alpha=0.3)

    # ── Panel B: Standard deviation vs size ───────────────────────────────────
    # Standard deviation measures spread. If variance compresses with diversity,
    # this line should slope downward.
    # We compute it for each individual community size (not just bins)
    # so we can see the smooth trend.
    size_stats = (df_comm_index
                  .groupby("community_size")["community_luminescence"]
                  .agg(["std", "count"])
                  .reset_index())

    # Only plot sizes with at least 5 communities (to avoid noisy estimates
    # from rare sizes).
    size_stats = size_stats[size_stats["count"] >= 5]

    # Scatter the actual std values.
    # The dot size (s=) scales with sample count — larger dot = more data = more reliable.
    axes[1].scatter(size_stats["community_size"], size_stats["std"],
                    s=size_stats["count"] * 2, color=PAL["specialist"],
                    alpha=0.7, edgecolors="white", linewidths=0.5,
                    label="Observed SD (dot size ∝ sample count)")

    # Add a smoothed trend line using a rolling average.
    # sort_values ensures the rolling window moves in order of community size.
    size_stats_sorted = size_stats.sort_values("community_size")
    # rolling(window=5, min_periods=1).mean() computes a 5-point moving average.
    # min_periods=1 means compute even if fewer than 5 points are available.
    rolling_std = size_stats_sorted["std"].rolling(window=5, min_periods=1).mean()
    axes[1].plot(size_stats_sorted["community_size"], rolling_std,
                 color=PAL["highlight"], lw=2.5, label="Smoothed trend")

    # Theoretical 1/√N expectation: if strains were independent (no interactions),
    # variance would shrink exactly as 1/√N. We overlay this as a reference.
    N_range = np.linspace(2, 50, 100)
    # We scale it to match the observed variance at small N.
    scale = size_stats_sorted["std"].iloc[:5].mean() * np.sqrt(
        size_stats_sorted["community_size"].iloc[:5].mean()
    )
    axes[1].plot(N_range, scale / np.sqrt(N_range),
                 color=PAL["neutral"], lw=1.5, ls="--",
                 label="1/√N null (if strains independent)")

    axes[1].set_title("B) Variance Compression vs Community Size\n"
                      "(decreasing SD = outcomes converging)")
    axes[1].set_xlabel("Community size (N)")
    axes[1].set_ylabel("Standard deviation of community luminescence")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    # ── Panel C: Percentile trends ────────────────────────────────────────────
    # Instead of just the mean, we track three lines:
    #   10th percentile (worst 10% of communities)
    #   50th percentile (median community)
    #   90th percentile (best 10% of communities)
    # If these three lines converge as size increases, outcomes are converging.
    binned_pct = (df_comm_index
                  .groupby("size_bin", observed=True)["community_luminescence"]
                  .quantile([0.10, 0.50, 0.90])
                  .unstack()
                  .reset_index())

    x_pos = np.arange(len(binned_pct))
    # Each quantile is a column; .quantile() with a list returns a DataFrame
    # with MultiIndex — .unstack() pivots quantile values to columns.
    axes[2].plot(x_pos, binned_pct[0.50], color=PAL["specialist"],
                 lw=2.5, marker="o", markersize=7,
                 markerfacecolor="white", markeredgewidth=2,
                 label="50th percentile (median)")
    axes[2].plot(x_pos, binned_pct[0.10], color=PAL["pathogen"],
                 lw=1.8, ls="--", marker="^", markersize=6,
                 label="10th percentile (worst 10%)")
    axes[2].plot(x_pos, binned_pct[0.90], color=PAL["generalist"],
                 lw=1.8, ls="--", marker="v", markersize=6,
                 label="90th percentile (best 10%)")
    axes[2].fill_between(x_pos, binned_pct[0.10], binned_pct[0.90],
                         alpha=0.10, color=PAL["specialist"])

    axes[2].axhline(0, color=PAL["neutral"], lw=1, ls=":")
    axes[2].set_xticks(x_pos)
    axes[2].set_xticklabels(labels)
    axes[2].set_title("C) Percentile Trends vs Size\n"
                      "(converging lines = outcomes becoming similar)")
    axes[2].set_xlabel("Community size bin")
    axes[2].set_ylabel("Community luminescence")
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "fig4_variance_compression.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  → fig4_variance_compression.png")


# =============================================================================
# FIGURE 5 — Uptake Rate Heatmap (who eats what?)
# =============================================================================

def fig5_uptake_heatmap(c_matrix, strain_type, OUTPUT_DIR,
                        N_RESOURCES, RESOURCE_NAMES, N_STRAINS):
    """
    Shows the uptake rate matrix for a sample of strains — which strains
    consume which nutrients, and at what rate.

    Panel A: Heatmap of uptake rates (strains × nutrients)
    Panel B: Active nutrient count distribution (sparsity by type)

    Why does this matter?
    Sparsity in the uptake matrix is what makes cross-feeding possible —
    specialists leave behind nutrients they can't consume, which generalists
    then pick up. This plot lets you visually verify the sparsity is realistic.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle("Figure 5 — Uptake Rate Matrix (who eats what?)\n"
                 "White = cannot consume this nutrient (masked out)",
                 fontsize=14, fontweight="bold", color=PAL["text"])

    # ── Panel A: Heatmap ──────────────────────────────────────────────────────
    # c_matrix rows are species (0=pathogen, 1..100=library), columns are nutrients.
    # We show a sample of 50 library strains (not the pathogen) for readability.
    # The first 25 specialists and first 25 generalists.
    n_specialists = sum(1 for t in strain_type[1:] if t == "specialist")
    spec_indices  = [i for i in range(1, N_STRAINS+1) if strain_type[i] == "specialist"][:25]
    gen_indices   = [i for i in range(1, N_STRAINS+1) if strain_type[i] == "generalist"][:25]
    sample_indices = spec_indices + gen_indices

    sample_matrix = c_matrix[sample_indices, :]
    sample_labels  = [f"{'Sp' if strain_type[i]=='specialist' else 'Gn'}{j+1}"
                      for j, i in enumerate(sample_indices)]

    # sns.heatmap draws a colour-coded grid.
    # cmap="YlOrRd" = yellow-orange-red (low to high).
    # vmin=0, vmax=1 sets the colour scale range.
    # linewidths=0.3 draws thin grid lines between cells.
    # annot=False skips printing numbers in each cell (too many cells).
    sns.heatmap(
        sample_matrix, ax=axes[0],
        cmap="YlOrRd", vmin=0, vmax=0.8,
        xticklabels=RESOURCE_NAMES,
        yticklabels=sample_labels,
        linewidths=0.3, linecolor="#f0f0f0",
        cbar_kws={"label": "Uptake rate c_iα  (0 = cannot consume)"}
    )
    axes[0].set_title("A) Uptake Rates — 50 sampled strains × 20 nutrients\n"
                      "(top 25 = specialists, bottom 25 = generalists)")
    axes[0].set_xlabel("Nutrient")
    axes[0].set_ylabel("Strain  (Sp=specialist, Gn=generalist)")
    axes[0].tick_params(axis="y", labelsize=7)
    axes[0].tick_params(axis="x", labelsize=8, rotation=45)

    # Colour the y-axis labels by type for easy visual separation.
    for tick, idx in zip(axes[0].get_yticklabels(), sample_indices):
        tick.set_color(PAL[strain_type[idx]])

    # ── Panel B: Active nutrient count distribution ───────────────────────────
    # For each strain, count how many nutrients have uptake > 0.
    # This shows how "wide" or "narrow" each type's diet is.
    # (c_matrix > 0).sum(axis=1) counts non-zero entries per row.
    active_counts = (c_matrix[1:, :] > 0).sum(axis=1)   # skip pathogen (row 0)

    for stype in ["specialist", "generalist"]:
        idx = [i for i, t in enumerate(strain_type[1:]) if t == stype]
        axes[1].hist([active_counts[i] for i in idx], bins=15,
                     color=PAL[stype], alpha=0.6,
                     label=f"{stype.capitalize()} (n={len(idx)})",
                     edgecolor="none")

    axes[1].set_title("B) Number of Active Nutrients per Strain\n"
                      "(specialists = narrow diet, generalists = broad)")
    axes[1].set_xlabel("Number of nutrients consumed (active uptake > 0)")
    axes[1].set_ylabel("Number of strains")
    axes[1].legend(fontsize=9)
    axes[1].grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "fig5_uptake_heatmap.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  → fig5_uptake_heatmap.png")


# =============================================================================
# FIGURE 6 — GO Feature Sparsity (what does the genome data look like?)
# =============================================================================

def fig6_go_features(go_matrix, df_mono, OUTPUT_DIR):
    """
    Visualises the gene-orthologue (GO) feature matrix — the input for GAT-GO.

    Panel A: Heatmap of GO features for a sample of strains (first 40 strains,
             first 100 GO features — full matrix is too large to display)
    Panel B: Distribution of how many strains carry each GO feature
             (should be heavily right-skewed — most features are rare)
    Panel C: ss_luminescence vs total GO feature count
             (does having more genes correlate with better suppression?)

    Why does this matter?
    This is the GAT-GO model's raw input. You want to confirm that the matrix
    is appropriately sparse (most entries are 0), that feature frequencies are
    realistic, and that there is at least some signal linking genomes to suppression.
    """
    fig, axes = plt.subplots(1, 3, figsize=(17, 6))
    fig.suptitle("Figure 6 — Gene Orthologue (GO) Feature Matrix\n"
                 "(input for GAT-GO model)",
                 fontsize=14, fontweight="bold", color=PAL["text"])

    # ── Panel A: Heatmap sample ───────────────────────────────────────────────
    # go_matrix is [100 strains × 1000 GO features].
    # We show only the first 40 strains and first 100 features — a window.
    sample_go    = go_matrix[:40, :100]
    strain_labels = [f"S{i+1:03d}" for i in range(40)]
    go_labels    = [f"GO{f+1}" for f in range(100)]

    sns.heatmap(
        sample_go, ax=axes[0],
        cmap="Blues", vmin=0, vmax=3,
        xticklabels=10,   # show every 10th label (not all 100) to avoid crowding
        yticklabels=strain_labels,
        linewidths=0,      # no grid lines — too many cells
        cbar_kws={"label": "Copy number (0 = absent, 1-3 = present)"}
    )
    axes[0].set_title("A) GO Matrix Sample\n"
                      "(40 strains × first 100 of 1,000 features)\n"
                      "Blue = gene present, White = absent")
    axes[0].set_xlabel("Gene orthologue (GO feature index)")
    axes[0].set_ylabel("Strain")
    axes[0].tick_params(axis="y", labelsize=7)

    # ── Panel B: Feature frequency distribution ───────────────────────────────
    # For each GO feature (column), count how many strains carry it.
    # (go_matrix > 0).sum(axis=0) counts non-zero entries per column.
    carrier_counts = (go_matrix > 0).sum(axis=0)   # shape [N_GO_FEATURES]

    axes[1].hist(carrier_counts, bins=30, color=PAL["specialist"],
                 alpha=0.75, edgecolor="white")
    # axvline at the median shows the typical feature frequency.
    axes[1].axvline(np.median(carrier_counts), color=PAL["highlight"],
                    lw=2, ls="--",
                    label=f"Median = {np.median(carrier_counts):.0f} strains")
    axes[1].set_title("B) How Many Strains Carry Each GO Feature?\n"
                      "(skewed right = most features are rare)")
    axes[1].set_xlabel("Number of strains carrying this GO feature")
    axes[1].set_ylabel("Number of GO features")
    axes[1].legend(fontsize=9)
    axes[1].grid(True, axis="y", alpha=0.3)

    # ── Panel C: Total GO count vs suppression ────────────────────────────────
    # Does having more genes (higher total GO count) make a strain a better
    # suppressor? This tests whether there is signal linking genomes to phenotype.
    total_go = (go_matrix > 0).sum(axis=1)   # total features per strain [100]

    for stype in ["specialist", "generalist"]:
        idx = df_mono[df_mono["strain_type"] == stype].index.tolist()
        axes[2].scatter(
            total_go[idx],
            df_mono.loc[idx, "ss_luminescence"],
            c=PAL[stype], s=50, alpha=0.7,
            edgecolors="white", linewidths=0.5,
            label=stype.capitalize()
        )

    axes[2].axhline(0, color=PAL["neutral"], lw=1.2, ls="--",
                    label="No suppression")
    axes[2].set_title("C) GO Feature Count vs Suppression Ability\n"
                      "(is having more genes better?)")
    axes[2].set_xlabel("Total GO features present (out of 1,000)")
    axes[2].set_ylabel("ss_luminescence (lower = better suppressor)")
    axes[2].legend(fontsize=9)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "fig6_go_features.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  → fig6_go_features.png")


# =============================================================================
# FIGURE 7 — Community Member Breakdown (who is in the communities?)
# =============================================================================

def fig7_community_breakdown(df_community, df_comm_index, OUTPUT_DIR):
    """
    Checks that communities were assembled correctly:

    Panel A: How often does each strain appear across all communities?
             (should be roughly equal — every strain gets fair representation)
    Panel B: Fraction of specialists vs generalists per size bin
             (should be roughly constant — assembly is random)
    Panel C: ss_luminescence of community members vs community outcome
             (communities with better individual strains should suppress more)

    Why does this matter?
    You want to confirm random assembly worked correctly and that individual
    strain quality has some (but not total) relationship to community outcome.
    """
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle("Figure 7 — Community Assembly Verification",
                 fontsize=14, fontweight="bold", color=PAL["text"])

    # ── Panel A: Strain appearance frequency ──────────────────────────────────
    # value_counts() counts how many times each strain_id appears across all
    # community member rows. Each appearance = one community that strain is in.
    strain_freq = df_community["strain_id"].value_counts().sort_values()

    # Colour bars by strain type (look up each strain's type).
    # We need a mapping from strain_id → strain_type.
    type_map = df_community.drop_duplicates("strain_id").set_index("strain_id")["strain_type"]
    bar_colors = [PAL.get(type_map.get(s, "neutral"), PAL["neutral"])
                  for s in strain_freq.index]

    axes[0].barh(range(len(strain_freq)), strain_freq.values,
                 color=bar_colors, edgecolor="none", alpha=0.8)
    axes[0].axvline(strain_freq.mean(), color=PAL["highlight"],
                    lw=2, ls="--",
                    label=f"Mean = {strain_freq.mean():.0f} appearances")
    axes[0].set_title("A) How Often Each Strain Appears\n"
                      "(should be roughly uniform = fair random assembly)")
    axes[0].set_xlabel("Number of communities strain appears in")
    axes[0].set_ylabel("Strain (sorted by frequency)")
    axes[0].set_yticks([])   # hide y-axis labels — too many strains
    axes[0].legend(fontsize=9)
    axes[0].grid(True, axis="x", alpha=0.3)

    # ── Panel B: Specialist fraction per size bin ─────────────────────────────
    # If assembly is truly random, specialist fraction should stay constant
    # at ~40% regardless of community size (since 40% of the library are specialists).
    bins   = [1, 4, 8, 15, 24, 35, 50]
    labels_b = ["2-4", "5-8", "9-15", "16-24", "25-35", "36-50"]

    df_community["size_bin"] = pd.cut(
        df_community["community_size"], bins=bins, labels=labels_b
    )

    # For each size bin, compute the fraction of member rows that are specialists.
    # (strain_type == "specialist") creates a True/False column.
    # .mean() of True/False = fraction that is True.
    spec_frac = (df_community
                 .groupby("size_bin", observed=True)
                 .apply(lambda g: (g["strain_type"] == "specialist").mean())
                 .reset_index())
    spec_frac.columns = ["size_bin", "specialist_fraction"]

    x_pos = np.arange(len(spec_frac))
    axes[1].bar(x_pos, spec_frac["specialist_fraction"],
                color=PAL["specialist"], alpha=0.75, edgecolor="white", width=0.6)
    # Reference line at 40% (the library-wide specialist fraction)
    axes[1].axhline(0.40, color=PAL["highlight"], lw=2, ls="--",
                    label="Library average (40%)")
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(labels_b)
    axes[1].set_ylim(0, 0.7)
    axes[1].set_title("B) Specialist Fraction per Size Bin\n"
                      "(should stay near 40% = random assembly)")
    axes[1].set_xlabel("Community size bin")
    axes[1].set_ylabel("Fraction of community that are specialists")
    axes[1].legend(fontsize=9)
    axes[1].grid(True, axis="y", alpha=0.3)

    # ── Panel C: Mean ss_luminescence of members vs community outcome ──────────
    # For each community, compute the mean of members' ss_luminescence.
    # Then plot this against the community's actual luminescence.
    # If this scatter is perfectly correlated (diagonal line), the GNN adds
    # nothing over a simple mean. If there's scatter, interactions matter.

    # groupby("community_id").mean() computes per-community averages.
    comm_mean_ss = (df_community
                    .groupby("community_id")["ss_luminescence"]
                    .mean()
                    .reset_index())
    comm_mean_ss.columns = ["community_id", "mean_ss_lum"]

    # merge() joins two DataFrames on a shared column (community_id here).
    # on="community_id" means match rows where community_id is the same.
    merged = comm_mean_ss.merge(
        df_comm_index[["community_id", "community_luminescence",
                       "community_size"]],
        on="community_id"
    )

    sc = axes[2].scatter(
        merged["mean_ss_lum"], merged["community_luminescence"],
        c=merged["community_size"], cmap="viridis_r",
        s=10, alpha=0.35, edgecolors="none"
    )
    plt.colorbar(sc, ax=axes[2], label="Community size")

    # Diagonal reference line: if community lum = mean ss_lum, it means
    # the community outcome equals the average individual outcome — no
    # interaction effect at all.
    lo = min(merged["mean_ss_lum"].min(), merged["community_luminescence"].min())
    hi = max(merged["mean_ss_lum"].max(), merged["community_luminescence"].max())
    axes[2].plot([lo, hi], [lo, hi], color=PAL["neutral"],
                 lw=1.5, ls="--", label="y = x (pure additive)")

    axes[2].set_title("C) Mean Individual Ability vs Community Outcome\n"
                      "(scatter above/below y=x = interaction effects)")
    axes[2].set_xlabel("Mean ss_luminescence of community members")
    axes[2].set_ylabel("Community luminescence (actual outcome)")
    axes[2].legend(fontsize=9)
    axes[2].grid(True, alpha=0.25)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "fig7_community_breakdown.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  → fig7_community_breakdown.png")


# ── Paper's exact size bins ───────────────────────────────────────────────────
# These come directly from the draft paper's figure labels and supplementary.
PAPER_BINS   = [2, 5, 9, 17, 33, 51]          # bin edges (51 so 50 is included)
PAPER_LABELS = ["2–4\n(Small)", "5–8", "9–16", "17–32", "33–50\n(Large)"]
 
# Colour each bin on a gradient from blue (small) to red (large),
# matching the paper's colour scheme in its community size plots.
BIN_COLORS = ["#1565C0", "#1E88E5", "#43A047", "#FB8C00", "#E53935"]
 
 
def fig8_community_size_distribution(df_comm_index, OUTPUT_DIR):
    """
    Generate a figure showing how communities are distributed across size bins
    that match the draft paper exactly.
 
    Parameters
    ----------
    df_comm_index : pd.DataFrame
        The community_index.csv DataFrame. Must have columns:
          - community_size  : int, number of strains in the community
          - community_luminescence : float, log-luminescence outcome
          - community_suppression  : float, 0-1 suppression score
 
    OUTPUT_DIR : str
        Folder to save the figure into.
    """
 
    # ── Bin the communities using the paper's exact boundaries ───────────────
    # pd.cut() assigns each community to a bin based on its size.
    # right=False means bins are [left, right) — so bin [2,5) contains 2,3,4
    # but NOT 5 (5 goes into the next bin [5,9)).
    # include_lowest=True ensures size=2 is captured in the first bin.
    df = df_comm_index.copy()
    df["size_bin"] = pd.cut(
        df["community_size"],
        bins=PAPER_BINS,
        labels=PAPER_LABELS,
        right=False,          # [left, right) — left-inclusive, right-exclusive
        include_lowest=True   # include the very first left edge (size = 2)
    )
 
    # Count how many communities fall into each bin.
    # value_counts() counts occurrences of each category.
    # reindex(PAPER_LABELS) ensures bins appear in order even if some are empty.
    bin_counts = df["size_bin"].value_counts().reindex(PAPER_LABELS, fill_value=0)
 
    # Compute per-bin statistics for the outcome (luminescence, suppression).
    # We want to show not just how many communities are in each bin, but also
    # what the typical outcome looks like — so the figure connects sample size
    # to biological result in one view.
    bin_stats = (df.groupby("size_bin", observed=True)
                   .agg(
                       count     = ("community_size",        "count"),
                       mean_lum  = ("community_luminescence","mean"),
                       std_lum   = ("community_luminescence","std"),
                       mean_supp = ("community_suppression", "mean"),
                       std_supp  = ("community_suppression", "std"),
                       min_size  = ("community_size",        "min"),
                       max_size  = ("community_size",        "max"),
                   )
                   .reindex(PAPER_LABELS)
                   .reset_index())
 
    # ── Figure layout ─────────────────────────────────────────────────────────
    # We use gridspec for fine control over panel sizes.
    # GridSpec(2, 3) creates a 2-row, 3-column grid.
    # We then span panels across multiple cells.
    fig = plt.figure(figsize=(18, 10), facecolor="#FAFAFA")
    fig.suptitle(
        "Figure 8 — Community Size Distribution\n"
        "Using the paper's exact size bins: Small (2–4), Medium (5–32), Large (33–50)",
        fontsize=14, fontweight="bold", color="#212121", y=0.98
    )
 
    # gridspec gives more control than plt.subplots() — we can make some panels
    # wider than others. width_ratios=[2,2,1] makes the first two panels wider.
    gs = plt.GridSpec(2, 3, figure=fig,
                      width_ratios=[2, 2, 1.5],
                      hspace=0.40, wspace=0.35)
 
    ax_bar    = fig.add_subplot(gs[0, 0])   # top-left:  bar chart of bin counts
    ax_hist   = fig.add_subplot(gs[0, 1])   # top-middle: raw histogram of sizes
    ax_pie    = fig.add_subplot(gs[0, 2])   # top-right:  pie chart
    ax_lum    = fig.add_subplot(gs[1, 0])   # bottom-left: lum per bin
    ax_supp   = fig.add_subplot(gs[1, 1])   # bottom-middle: suppression per bin
    ax_table  = fig.add_subplot(gs[1, 2])   # bottom-right: summary table
 
    x_pos = np.arange(len(PAPER_LABELS))
 
    # ─────────────────────────────────────────────────────────────────────────
    # Panel A — Bar chart: communities per bin
    # ─────────────────────────────────────────────────────────────────────────
    bars = ax_bar.bar(x_pos, bin_counts.values,
                      color=BIN_COLORS, edgecolor="white",
                      linewidth=1.5, width=0.65, alpha=0.88)
 
    # Label each bar with its count AND percentage of total.
    total = bin_counts.sum()
    for bar, count in zip(bars, bin_counts.values):
        pct = 100 * count / total if total > 0 else 0
        # Text inside the bar (the count number).
        ax_bar.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() / 2,
            f"{count:,}",
            ha="center", va="center",
            fontsize=11, fontweight="bold", color="white"
        )
        # Text above the bar (the percentage).
        ax_bar.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 5,
            f"{pct:.1f}%",
            ha="center", va="bottom",
            fontsize=9, color="#212121"
        )
 
    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(PAPER_LABELS, fontsize=9)
    ax_bar.set_title("A) Communities per Size Bin\n(paper's exact bins)",
                     fontweight="bold")
    ax_bar.set_xlabel("Community size bin")
    ax_bar.set_ylabel("Number of communities")
    ax_bar.set_ylim(0, bin_counts.max() * 1.20)
    ax_bar.grid(True, axis="y", alpha=0.3)
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)
 
    # Add a shaded annotation separating "small" from "large" visually.
    # axvspan shades a vertical band between x1 and x2.
    ax_bar.axvspan(-0.5, 0.5, alpha=0.06, color=BIN_COLORS[0],
                   label="Small (paper: N≤8)")
    ax_bar.axvspan(3.5, 4.5, alpha=0.06, color=BIN_COLORS[4],
                   label="Large (paper: N≥20)")
    ax_bar.legend(fontsize=7, loc="upper right")
 
    # ─────────────────────────────────────────────────────────────────────────
    # Panel B — Raw histogram of individual community sizes
    # ─────────────────────────────────────────────────────────────────────────
    # This shows the full distribution before binning — you can see which
    # exact sizes (2, 3, 4, ..., 50) were generated and how often.
    # np.arange(2, 52) gives bin edges at each integer from 2 to 51,
    # so each bar is exactly one community size.
    counts_per_size, edges, patches = ax_hist.hist(
        df["community_size"],
        bins=np.arange(2, 52),   # one bar per integer size
        color="#90CAF9",          # light blue fill
        edgecolor="white",
        linewidth=0.5,
        alpha=0.9
    )
 
    # Colour each bar according to which paper bin it falls into.
    # patches is a list of Rectangle objects, one per bar.
    # edges[i] is the left edge of bar i, which equals the community size.
    for patch, left_edge in zip(patches, edges[:-1]):
        size = int(left_edge)
        if   2  <= size <= 4:   patch.set_facecolor(BIN_COLORS[0])
        elif 5  <= size <= 8:   patch.set_facecolor(BIN_COLORS[1])
        elif 9  <= size <= 16:  patch.set_facecolor(BIN_COLORS[2])
        elif 17 <= size <= 32:  patch.set_facecolor(BIN_COLORS[3])
        elif 33 <= size <= 50:  patch.set_facecolor(BIN_COLORS[4])
 
    # Add vertical dashed lines at the paper's bin boundaries so readers
    # can see exactly where each bin starts and ends.
    for boundary, label_txt in zip(PAPER_BINS[1:-1], ["5", "9", "17", "33"]):
        ax_hist.axvline(boundary, color="#424242", lw=1.2, ls="--", alpha=0.6)
        ax_hist.text(boundary + 0.3,
                     counts_per_size.max() * 0.92,
                     label_txt,
                     fontsize=8, color="#424242", rotation=0)
 
    ax_hist.set_title("B) Raw Size Distribution\n(coloured by paper bin)",
                      fontweight="bold")
    ax_hist.set_xlabel("Community size (exact number of strains)")
    ax_hist.set_ylabel("Number of communities")
    ax_hist.grid(True, axis="y", alpha=0.3)
    ax_hist.spines["top"].set_visible(False)
    ax_hist.spines["right"].set_visible(False)
 
    # Custom legend for bin colours.
    legend_patches = [
        mpatches.Patch(color=BIN_COLORS[i], label=PAPER_LABELS[i].replace("\n", " "))
        for i in range(len(PAPER_LABELS))
    ]
    ax_hist.legend(handles=legend_patches, fontsize=7, loc="upper right")
 
    # ─────────────────────────────────────────────────────────────────────────
    # Panel C — Pie chart: proportion small vs large
    # ─────────────────────────────────────────────────────────────────────────
    # The paper specifically contrasts "small (N≤8)" vs "large (N≥20)".
    # We create three slices: small, medium (9-32), large.
    n_small  = len(df[df["community_size"].between(2, 8)])
    n_medium = len(df[df["community_size"].between(9, 32)])
    n_large  = len(df[df["community_size"].between(33, 50)])
 
    pie_sizes  = [n_small, n_medium, n_large]
    pie_labels = [f"Small\n(N=2–8)\nn={n_small}",
                  f"Medium\n(N=9–32)\nn={n_medium}",
                  f"Large\n(N=33–50)\nn={n_large}"]
    pie_colors = [BIN_COLORS[0], BIN_COLORS[2], BIN_COLORS[4]]
 
    # explode slightly separates each slice from the centre.
    # The small and large slices are pulled out more to emphasise them
    # (they are the focus of the paper's cross-scale transfer analysis).
    wedges, texts, autotexts = ax_pie.pie(
        pie_sizes,
        labels=pie_labels,
        colors=pie_colors,
        explode=(0.05, 0.0, 0.05),   # pull small and large out slightly
        autopct="%1.1f%%",           # show percentage inside each slice
        pctdistance=0.65,            # how far from centre the % label sits
        startangle=90,               # start at top (12 o'clock position)
        wedgeprops=dict(edgecolor="white", linewidth=2)
    )
 
    # Make the percentage text bold and readable.
    for autotext in autotexts:
        autotext.set_fontsize(9)
        autotext.set_fontweight("bold")
        autotext.set_color("white")
 
    ax_pie.set_title("C) Small vs Medium vs Large\n(paper's cross-scale boundary)",
                     fontweight="bold")
 
    # ─────────────────────────────────────────────────────────────────────────
    # Panel D — Mean luminescence per bin (outcome by bin)
    # ─────────────────────────────────────────────────────────────────────────
    # This connects the sample size question (how many?) to the biological
    # question (what do they achieve?). Each bin's mean luminescence should
    # decrease (get more negative) as size increases.
    valid = bin_stats.dropna(subset=["mean_lum"])
 
    ax_lum.bar(range(len(valid)), valid["mean_lum"],
               color=[BIN_COLORS[i] for i in range(len(valid))],
               edgecolor="white", linewidth=1.2, width=0.65, alpha=0.85)
 
    # Error bars show ± 1 standard deviation.
    ax_lum.errorbar(range(len(valid)), valid["mean_lum"],
                    yerr=valid["std_lum"],
                    fmt="none", color="#212121", lw=1.5, capsize=5)
 
    ax_lum.axhline(0, color="#78909C", lw=1.2, ls="--",
                   label="Pathogen solo reference")
    ax_lum.set_xticks(range(len(valid)))
    ax_lum.set_xticklabels(PAPER_LABELS[:len(valid)], fontsize=9)
    ax_lum.set_title("D) Mean Suppression per Bin\n(should decrease = more suppression with size)",
                     fontweight="bold")
    ax_lum.set_xlabel("Community size bin")
    ax_lum.set_ylabel("Mean community luminescence\n(mean ± SD)")
    ax_lum.legend(fontsize=8)
    ax_lum.grid(True, axis="y", alpha=0.3)
    ax_lum.spines["top"].set_visible(False)
    ax_lum.spines["right"].set_visible(False)
 
    # ─────────────────────────────────────────────────────────────────────────
    # Panel E — Suppression score with connecting line
    # ─────────────────────────────────────────────────────────────────────────
    # Same as D but using the 0-1 suppression score, with a line connecting
    # the means to show the trend more clearly (like the paper's Figure 1B).
    ax_supp.bar(range(len(valid)), valid["mean_supp"],
                color=[BIN_COLORS[i] for i in range(len(valid))],
                edgecolor="white", linewidth=1.2, width=0.65, alpha=0.50)
 
    # Plot the means connected by a line on top of the bars.
    ax_supp.plot(range(len(valid)), valid["mean_supp"],
                 color="#212121", lw=2.2, marker="D",
                 markersize=8, markerfacecolor="white",
                 markeredgecolor="#212121", markeredgewidth=2,
                 label="Mean suppression score", zorder=5)
 
    # Shaded band for ± 1 std.
    ax_supp.fill_between(
        range(len(valid)),
        valid["mean_supp"] - valid["std_supp"],
        valid["mean_supp"] + valid["std_supp"],
        alpha=0.12, color="#212121"
    )
 
    ax_supp.set_xticks(range(len(valid)))
    ax_supp.set_xticklabels(PAPER_LABELS[:len(valid)], fontsize=9)
    ax_supp.set_ylim(-0.05, 1.10)
    ax_supp.set_title("E) Suppression Score per Bin\n(0 = no suppression, 1 = full)",
                     fontweight="bold")
    ax_supp.set_xlabel("Community size bin")
    ax_supp.set_ylabel("Suppression score (mean ± SD)")
    ax_supp.legend(fontsize=8)
    ax_supp.grid(True, axis="y", alpha=0.3)
    ax_supp.spines["top"].set_visible(False)
    ax_supp.spines["right"].set_visible(False)
 
    # ─────────────────────────────────────────────────────────────────────────
    # Panel F — Summary table
    # ─────────────────────────────────────────────────────────────────────────
    # A clean summary table showing all the numbers in one place.
    # This is the most useful panel for deciding whether you have enough data
    # in each bin before training the GNN.
    ax_table.axis("off")   # turn off the axis — we're drawing a table, not a plot
 
    table_data = []
    for _, row in bin_stats.iterrows():
        if pd.isna(row["count"]):
            continue
        table_data.append([
            str(row["size_bin"]).replace("\n", " "),
            f"{int(row['count']):,}",
            f"{100*int(row['count'])/total:.1f}%",
            f"{row['mean_lum']:.3f}",
            f"{row['mean_supp']:.3f}",
        ])
 
    col_headers = ["Size Bin", "Count", "% Total",
                   "Mean Lum", "Mean Supp"]
 
    # ax.table() draws a table directly on a matplotlib axis.
    # cellText = the data rows, colLabels = the header row.
    # cellLoc="center" centres text in each cell.
    # loc="center" centres the table on the axis.
    tbl = ax_table.table(
        cellText=table_data,
        colLabels=col_headers,
        cellLoc="center",
        loc="center",
        bbox=[0, 0.05, 1, 0.90]   # [left, bottom, width, height] in axis fraction
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
 
    # Style the header row differently (darker background).
    for col_idx in range(len(col_headers)):
        tbl[(0, col_idx)].set_facecolor("#1565C0")
        tbl[(0, col_idx)].set_text_props(color="white", fontweight="bold")
 
    # Colour each data row by its corresponding bin colour.
    for row_idx, color in enumerate(BIN_COLORS[:len(table_data)], start=1):
        for col_idx in range(len(col_headers)):
            tbl[(row_idx, col_idx)].set_facecolor(color + "22")  # "22" = 13% opacity hex
 
    ax_table.set_title("F) Summary Table", fontweight="bold", pad=10)
 
    # ── Save ─────────────────────────────────────────────────────────────────
    path = os.path.join(OUTPUT_DIR, "fig8_community_size_distribution.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  → fig8_community_size_distribution.png")


# =============================================================================
# MASTER FUNCTION — call this from your main script
# =============================================================================

def make_figures(df_community, df_mono, df_comm_index,
                 go_matrix, c_matrix, strain_type, OUTPUT_DIR,
                 N_RESOURCES=20, RESOURCE_NAMES=None, N_STRAINS=100):
    """
    Generate all 7 figures.

    Parameters
    ----------
    df_community  : pd.DataFrame  from generate_dataset()
    df_mono       : pd.DataFrame  from generate_dataset()
    df_comm_index : pd.DataFrame  from generate_dataset()
    go_matrix     : ndarray [N_STRAINS, N_GO_FEATURES]  from build_strain_library()
    c_matrix      : ndarray [N_STRAINS+1, N_RESOURCES]  from build_strain_library()
    strain_type   : list[str]                            from build_strain_library()
    OUTPUT_DIR    : str  path to save figures

    Call at the end of __main__ like:
        make_figures(df_community, df_mono, df_comm_index,
                     go_matrix, c_matrix, strain_type, OUTPUT_DIR)
    """
    if RESOURCE_NAMES is None:
        RESOURCE_NAMES = [f"c{k+1}" for k in range(N_RESOURCES)]

    print("\n[Figures] Generating visualisations …")
    fig1_strain_library(df_mono, OUTPUT_DIR)
    fig2_single_strain_suppression(df_mono, OUTPUT_DIR)
    fig3_community_vs_size(df_comm_index, OUTPUT_DIR)
    fig4_variance_compression(df_comm_index, OUTPUT_DIR)
    fig5_uptake_heatmap(c_matrix, strain_type, OUTPUT_DIR,
                        N_RESOURCES, RESOURCE_NAMES, N_STRAINS)
    fig6_go_features(go_matrix, df_mono, OUTPUT_DIR)
    fig7_community_breakdown(df_community, df_comm_index, OUTPUT_DIR)
    fig8_community_size_distribution(df_comm_index, OUTPUT_DIR)
    print(f"\n  All figures saved to: {OUTPUT_DIR}")




In [9]:
"""
run_figures.py
==============
Run this file directly AFTER you have already generated the dataset.
It loads the saved CSV files and generates all 8 figures.

Make sure this file is in the SAME folder as:
  - community_figures.py
  - fig8_community_size_distribution.py

And make sure OUTPUT_DIR below points to wherever your CSVs were saved.
"""

import numpy as np
import pandas as pd
import os


# ─────────────────────────────────────────────────────────────────────────────
# EDIT THIS to match wherever your CSV files were saved
# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = r"C:\Users\Pratham Shah\Downloads\Document dataset"

# ─────────────────────────────────────────────────────────────────────────────
# These should match what you used when generating the dataset
# ─────────────────────────────────────────────────────────────────────────────
SEED          = 42
N_STRAINS     = 100
N_RESOURCES   = 20
N_GO_FEATURES = 1000
N_GENES_REAL  = 8
RESOURCE_NAMES = [f"c{k+1}" for k in range(N_RESOURCES)]

FRAC_SPECIALIST = 0.40
N_SPECIALIST    = int(N_STRAINS * FRAC_SPECIALIST)

UPTAKE_SPARSITY = {
    "pathogen"  : (0.30, 0.55),
    "specialist": (0.10, 0.25),
    "generalist": (0.40, 0.65),
}
CROSSFEED_OUT_MIN = 2
CROSSFEED_OUT_MAX = 6


# =============================================================================
# STEP 1 — Load the CSV files you already generated
# =============================================================================

print("=" * 60)
print("  LOADING SAVED DATASETS")
print("=" * 60)

path_comm_nogo = os.path.join(OUTPUT_DIR, "community_dataset_noGO.csv")
path_mono      = os.path.join(OUTPUT_DIR, "monoculture_dataset.csv")
path_comm_idx  = os.path.join(OUTPUT_DIR, "community_index.csv")

# pd.read_csv() reads a CSV file from disk into a DataFrame.
# This is the reverse of df.to_csv() — it reconstructs the table.
df_community  = pd.read_csv(path_comm_nogo)
df_mono       = pd.read_csv(path_mono)
df_comm_index = pd.read_csv(path_comm_idx)

print(f"  community_dataset_noGO : {df_community.shape}")
print(f"  monoculture_dataset    : {df_mono.shape}")
print(f"  community_index        : {df_comm_index.shape}")


# =============================================================================
# STEP 2 — Reconstruct the strain library (needed for Fig 5 and Fig 6)
#
# The figures that need c_matrix, strain_type, and go_matrix cannot read
# these from the CSV directly (they were numpy arrays, not saved as CSVs).
# So we rebuild them from scratch using the SAME seed (SEED=42) which
# guarantees we get the EXACT same random numbers as the original run.
# =============================================================================

print("\n  Reconstructing strain library from seed (for Figs 5 & 6) ...")

rng = np.random.default_rng(seed=SEED)

N_TOTAL = N_STRAINS + 1   # pathogen + library

# ── Rebuild genes and weight matrix (same seed = same values) ────────────────
genes = rng.integers(0, 2, size=(N_TOTAL, N_GENES_REAL))
W     = rng.normal(0.0, 0.5, size=(N_GENES_REAL, N_RESOURCES))

c_matrix    = np.zeros((N_TOTAL, N_RESOURCES))
uptake_mask = np.zeros((N_TOTAL, N_RESOURCES), dtype=bool)
strain_type = []
preferred_nutrient = []

for i in range(N_TOTAL):
    if i == 0:
        stype = "pathogen"
        pref  = -1
    elif 1 <= i <= N_SPECIALIST:
        stype = "specialist"
        pref  = (i - 1) % N_RESOURCES
    else:
        stype = "generalist"
        pref  = -1

    strain_type.append(stype)
    preferred_nutrient.append(pref)

    lo, hi   = UPTAKE_SPARSITY[stype]
    density  = rng.uniform(lo, hi)
    n_active = max(1, int(round(density * N_RESOURCES)))
    active_indices = rng.choice(N_RESOURCES, size=n_active, replace=False).tolist()

    if pref >= 0 and pref not in active_indices:
        active_indices[0] = pref

    uptake_mask[i, active_indices] = True

    if stype == "specialist":
        bias = rng.uniform(-2.0, -1.0, size=N_RESOURCES)
        bias[pref] += rng.uniform(2.5, 4.0)
    else:
        bias = rng.uniform(-0.8, -0.3, size=N_RESOURCES)

    raw      = genes[i].astype(float) @ W + bias
    c_full   = 1.0 / (1.0 + np.exp(-raw))
    c_matrix[i] = c_full * uptake_mask[i]

# maintenance and growth (consume rng state — must match original order)
m = rng.uniform(0.2, 0.6, size=N_TOTAL)
g = rng.uniform(0.8, 1.2, size=N_TOTAL)

# ── Rebuild cross-feeding matrices (to keep rng state in sync) ───────────────
# We need to call this to advance the rng to the same state as the
# original run — otherwise go_matrix will be wrong.
def _rebuild_crossfeed(N_TOTAL, strain_type, rng):
    D_list = []
    for i in range(N_TOTAL):
        D_i = np.zeros((N_RESOURCES, N_RESOURCES))
        for beta in range(N_RESOURCES):
            n_out = int(rng.integers(CROSSFEED_OUT_MIN, CROSSFEED_OUT_MAX + 1))
            active_alpha = rng.choice(N_RESOURCES, size=n_out, replace=False)
            if strain_type[i] == "specialist":
                pref_i = (i - 1) % N_RESOURCES
                alpha_vec = np.ones(n_out)
                for j, a in enumerate(active_alpha):
                    if a == pref_i and beta != pref_i:
                        alpha_vec[j] = rng.uniform(2.0, 5.0)
                    else:
                        alpha_vec[j] = rng.uniform(0.5, 2.0)
            elif strain_type[i] == "pathogen":
                alpha_vec = rng.uniform(0.5, 3.0, size=n_out)
            else:
                alpha_vec = rng.uniform(0.8, 1.5, size=n_out)
            probs = rng.dirichlet(alpha_vec)
            D_i[active_alpha, beta] = probs
        D_list.append(D_i)
    return D_list

_ = _rebuild_crossfeed(N_TOTAL, strain_type, rng)   # advance rng state only

# ── Rebuild GO matrix ─────────────────────────────────────────────────────────
functional_genes = genes[1:]   # exclude pathogen row
go_matrix = np.zeros((N_STRAINS, N_GO_FEATURES), dtype=np.int8)

for f in range(N_GO_FEATURES):
    frac       = rng.beta(0.5, 4.0)
    n_carriers = max(1, int(round(frac * N_STRAINS)))
    carriers   = rng.choice(N_STRAINS, size=n_carriers, replace=False)
    copy_nums  = rng.choice([1, 1, 1, 2, 3], size=n_carriers)
    go_matrix[carriers, f] = copy_nums

    if rng.random() < 0.20:
        linked_gene = int(rng.integers(0, N_GENES_REAL))
        has_gene    = np.where(functional_genes[:, linked_gene] == 1)[0]
        extra_carriers = rng.choice(
            has_gene,
            size=min(len(has_gene) // 4, max(1, n_carriers // 2)),
            replace=False
        )
        go_matrix[extra_carriers, f] = np.maximum(go_matrix[extra_carriers, f], 1)

print(f"  Strain library reconstructed — c_matrix: {c_matrix.shape}, "
      f"go_matrix: {go_matrix.shape}")


# =============================================================================
# STEP 3 — Generate all figures
# =============================================================================

print("\n" + "=" * 60)
print("  GENERATING FIGURES")
print("=" * 60)

make_figures(
    df_community   = df_community,
    df_mono        = df_mono,
    df_comm_index  = df_comm_index,
    go_matrix      = go_matrix,
    c_matrix       = c_matrix,
    strain_type    = strain_type,
    OUTPUT_DIR     = OUTPUT_DIR,
    N_RESOURCES    = N_RESOURCES,
    RESOURCE_NAMES = RESOURCE_NAMES,
    N_STRAINS      = N_STRAINS,
)

fig8_community_size_distribution(df_comm_index, OUTPUT_DIR)

print("\n" + "=" * 60)
print(f"  All figures saved to: {OUTPUT_DIR}")
print("=" * 60)

  LOADING SAVED DATASETS
  community_dataset_noGO : (39362, 62)
  monoculture_dataset    : (100, 1058)
  community_index        : (1500, 6)

  Reconstructing strain library from seed (for Figs 5 & 6) ...
  Strain library reconstructed — c_matrix: (101, 20), go_matrix: (100, 1000)

  GENERATING FIGURES

[Figures] Generating visualisations …
  → fig1_strain_library.png
  → fig2_single_strain_suppression.png
  → fig3_community_vs_size.png
  → fig4_variance_compression.png
  → fig5_uptake_heatmap.png
  → fig6_go_features.png
  → fig7_community_breakdown.png
  → fig8_community_size_distribution.png

  All figures saved to: C:\Users\Pratham Shah\Downloads\Document dataset
  → fig8_community_size_distribution.png

  All figures saved to: C:\Users\Pratham Shah\Downloads\Document dataset
